### Introduction to Quantum Mechanics, Second Edition | David J. Griffiths | Solutions and Code by David A. Lee
### Chapter 2: Time-Independent Schrödinger Equation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scienceplots

import sys; from pathlib import Path
# plotstyle.py sits in a parent folder of this notebook: find it instead of hardcoding a path
sys.path.insert(0, str(next(p for p in Path.cwd().parents if (p / "plotstyle.py").is_file())))
from plotstyle import THEME, C, FG, savefig, tint

In [ ]:
def style_ax(ax, xlabel='$x$', ylabel='$y$', xlim=None, ylim=None,
             xticks=None, xticklabels=None, yticks=None, yticklabels=None,
             color=FG, fontsize=16):
    """Apply shared plot attributes to one or more axes."""
    axes = np.atleast_1d(ax)
    for a in axes:
        a.set_xlabel(xlabel, color=color, fontsize=fontsize)
        a.set_ylabel(ylabel, color=color, fontsize=fontsize)
        a.tick_params(colors=color)
        a.tick_params(labelsize=fontsize)
        a.tick_params(which='minor', top=False, bottom=False, left=False, right=False)
        a.grid(False)
        if xlim is not None:
            a.set_xlim(xlim)
        if ylim is not None:
            a.set_ylim(ylim)
        if xticks is not None:
            a.set_xticks(xticks)
        if xticklabels is not None:
            a.set_xticklabels(xticklabels, color=color)
        if yticks is not None:
            a.set_yticks(yticks)
        if yticklabels is not None:
            a.set_yticklabels(yticklabels, color=color)

Question 2.6

In [ ]:
# |Psi(x,0)|^2 = (1/2)[psi_1^2 + psi_2^2 + 2 psi_1 psi_2 cos(phi)],  psi_n = sqrt(2/a) sin(n pi x/a)  (infinite square well).
# <x>(0) = a/2 - (16 a / 9 pi^2) cos(phi):  phi = 0 leans left, pi/2 is centered on average, pi leans right.
# Take a = 1 and label ticks symbolically so the sketch holds for any a > 0.
a = 1.0
x = np.linspace(0, a, 801)
psi1 = np.sqrt(2/a)*np.sin(np.pi*x/a)
psi2 = np.sqrt(2/a)*np.sin(2*np.pi*x/a)

phis = np.array([0, np.pi/2, np.pi])
rho = 0.5*(psi1**2 + psi2**2 + 2*psi1*psi2*np.cos(phis)[:, None])   # one row per phi
assert np.allclose(rho.sum(axis=1)*(x[1] - x[0]), 1.0)              # each normalized (endpoints are 0)
xmean = (rho*x).sum(axis=1)*(x[1] - x[0])
assert np.allclose(xmean, a/2 - 16*a/(9*np.pi**2)*np.cos(phis))      # matches the closed form

fig, ax = plt.subplots(figsize=(10, 4))

for r, xm, color, name in zip(rho, xmean, [C.cyan, C.green, C.magenta], ['0', r'\pi/2', r'\pi']):
    ax.plot(x, r, color=color, linewidth=2.5,
            label=rf'$\phi = {name}$, $\langle x \rangle = {xm/a:.3f}\,a$')
ax.vlines(a/2, 0, 1.15*rho.max(), colors=FG, linestyles='--', linewidths=1.2)

style_ax(ax, xlabel='$x$', ylabel=r'$|\Psi(x, 0)|^2$', xlim=(0, a), ylim=(0, 1.15*rho.max()),
         xticks=[0, a/2, a], xticklabels=['$0$', '$a/2$', '$a$'],
         yticks=[0, 2/a], yticklabels=['$0$', '$2/a$'])
ax.legend(frameon=True, fontsize=14, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=3)

savefig(fig, "chap2ex2.6.pdf")
plt.show()


Question 2.7

In [ ]:
# Psi(x,0) = A x on [0, a/2], A (a - x) on [a/2, a]: a tent peaking at Psi(a/2, 0) = A a/2.
# Normalization: |A|^2 a^3 / 12 = 1  =>  A = 2 sqrt(3) / a^(3/2), so the peak is sqrt(3/a).
# Take a = 1 and label ticks symbolically so the sketch holds for any a > 0.
a = 1.0
A = 2*np.sqrt(3)/a**1.5
assert np.isclose(A**2*a**3/12, 1.0)          # normalized

x = np.array([0, a/2, a])                     # piecewise linear: the three corners are the whole curve
psi = A*np.minimum(x, a - x)
assert np.isclose(psi[1], np.sqrt(3/a))

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(x, psi, color=C.cyan, linewidth=2.5)
ax.vlines(a/2, 0, psi[1], colors=FG, linestyles='--', linewidths=1.2)

style_ax(ax, xlabel='$x$', ylabel=r'$\Psi(x, 0)$', xlim=(0, a), ylim=(0, 1.15*psi[1]),
         xticks=[0, a/2, a], xticklabels=['$0$', '$a/2$', '$a$'],
         yticks=[0, psi[1]], yticklabels=['$0$', r'$\sqrt{3/a}$'])

savefig(fig, "chap2ex2.7a.pdf")
plt.show()

Question 2.10

In [ ]:
# Harmonic oscillator, xi = sqrt(m omega / hbar) x, in units of (m omega / pi hbar)^(1/4):
# psi_0 = e^{-xi^2/2},  psi_1 = sqrt(2) xi e^{-xi^2/2},  psi_2 = (2 xi^2 - 1)/sqrt(2) e^{-xi^2/2}.
# psi_n has n nodes and parity (-1)^n.
xi = np.linspace(-5, 5, 1001)
gauss = np.exp(-xi**2/2)
psi = np.array([np.ones_like(xi), np.sqrt(2)*xi, (2*xi**2 - 1)/np.sqrt(2)])*gauss   # one row per n

# orthonormal: integral of psi_m psi_n d(xi) = sqrt(pi) delta_mn in these units (tails are ~1e-11 at |xi| = 5)
assert np.allclose(psi @ psi.T*(xi[1] - xi[0]), np.sqrt(np.pi)*np.eye(3), atol=1e-8)

fig, ax = plt.subplots(figsize=(10, 4))

for n, (p, color) in enumerate(zip(psi, [C.cyan, C.green, C.magenta])):
    ax.plot(xi, p, color=color, linewidth=2.5, label=rf'$\psi_{n}$')
ax.axhline(0, color=FG, linestyle='--', linewidth=1.2)

style_ax(ax, xlabel=r'$\xi = \sqrt{m\omega/\hbar}\,x$', ylabel=r'$\psi_n \,/\, (m\omega/\pi\hbar)^{1/4}$',
         xlim=(-5, 5), ylim=(-1.15, 1.15), xticks=np.arange(-4, 5, 2), yticks=[-1, 0, 1])
ax.legend(frameon=True, fontsize=14, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=3)

savefig(fig, "chap2ex2.10b.pdf")
plt.show()